* AIC/BIC pénalisent injustement les paramètres bayésiens
* possibilité de comparer les modèles sur des sous-échantillons de pays à condition de: garder la meme taille d'échantillon; éviter le biais d'échantillonage en incluant des pays instables aussi. OK
* PSIS-LOO: 


EMV: $\hat{\theta} = \arg\max P(Y|\theta)$.  
Complexité du système: toute une masse de proba, une distribution entière, pas un point singulier (le sommet d'une colline). 
Architecture ZTNB+AR(1): paysage très complexe, un point unique ne peut pas représenter tout le système. 
AIC ($-2 \ln(L) + 2k$) et BIC ($-2 \ln(L) + k \ln(n)$) : considère la complexité d'un paramètre comme l'entier brut "k".   
Exemple: $\alpha_i \sim \mathcal{N}(\mu_{em}, \tau_{em})$.  
Si  $\tau_{em} \to \infty$, le BIC est valide: chaque paramètre est indépendant, donc 190 paramètres $\alpha_i$, k=190. 
Si  $\tau_{em}$ est fini ou proche de zéro, les paramètres $\alpha_i$ sont contraints, leur vraie complexité n'est pas de 1, le BIC/AIC pénalisent injustement les 190 paramètres $\alpha_i$ comme 190 paramètres libres. En plus, $\tau_em$ est appris conjointement.  



Voir la valeur fractionnaire retournée par ArviZ p_loo (paramètres effectifs ($\alpha_i, \gamma_j$) ) et vérifier que < 380 comme considérer par AIC/BIC

Ensuite passer à PSIS-LOO, rapidement, car s'avère pas bon non plus. 


### ZTNB: 

NB type 2 (version mélange Poisson-Gamma) non tronquée en zéro: 
$P(Y = y \mid \mu, \phi) = \frac{\Gamma(y + \phi)}{\Gamma(\phi) y!} \left(\frac{\phi}{\mu + \phi}\right)^\phi \left(\frac{\mu}{\mu + \phi}\right)^y$. 

tronquée en zéro: 
$f_{\text{ZTNB}}(y \mid \mu, \phi) = \frac{P(Y = y \mid \mu, \phi)}{1 - p_0} = \frac{\Gamma(y + \phi)}{\Gamma(\phi) y! (1 - p_0)} \left(\frac{\phi}{\mu + \phi}\right)^\phi \left(\frac{\mu}{\mu + \phi}\right)^y$. 

Dans l'idéal: 
$Y \mid \lambda \sim \text{Poisson}(\lambda)$

Mais on ne connaît pas $\lambda$. 

Dans la réalité,  $X\beta$ ne produit qu'une espérance macro $\mu$. Le taux réel $\lambda$ varie autour de $\mu$.  
On introduit donc $\nu$, tel que $\lambda = \mu \nu$.  
Pour garantir que $\lambda$ reste strictement >0, on assigne à ce choc $\nu$ une Gamma d'espérance $1$ et de variance $1/\phi$.
Le paramètre $\lambda$ devient une variable aléatoire continue distribuée selon une Gamma de moyenne $\mu$ et de paramètre de forme $\phi$ :
$\lambda \sim \text{Gamma}\left(\text{forme} = \phi, \text{taux} = \frac{\phi}{\mu}\right)$




$$P(Y = y \mid \mu, \phi) = \int_0^\infty \underbrace{P(Y = y \mid \lambda)}_{\text{Poisson}} \times \underbrace{f(\lambda \mid \mu, \phi)}_{\text{Gamma}} d\lambda$$

Esperance: 

$\mathbb{E}[Y \mid Y > 0] = \frac{\mu}{1 - \left(\frac{\phi}{\mu + \phi}\right)^\phi} = \frac{\mu}{1 - p_0}$. 

Variance: 

$\text{Var}(Y \mid Y > 0) = \frac{\mu + \frac{\mu^2}{\phi}}{1 - p_0} - \frac{\mu^2 p_0}{(1 - p_0)^2}$. 

Phi: paramètre de dispersion. Mélange poisson gamma: le taux $\lambda$ de la Poisson est traité comme une variable aléatoire suivant une Gamma, $\phi$ est le paramètre de forme de cette Gamma sous-jacente. Si $\phi \to \infty$ , $$p_0 = \left( \frac{\phi}{\mu + \phi} \right)^\phi = \left( 1 + \frac{\mu}{\phi} \right)^{-\phi}$$ et la variance converge vers l'espérance (de la loi tronquée) 


Pour visualiser (l'approximation tient pour juste visualiser, mais ce n'est pas la variance exacte de la distrib tronquée en zéro): 
$\text{Var}(Y) \approx \mu + \frac{\mu^2}{\phi}$ 

La variance est sur-dispersée quadratiquement, c'est parfait pour modéliser les flux migratoires où V(Y)>>E(Y), alors qu'une Poisson par exemple imposerait V(Y)=E(Y).   
Si on observe juste la variance, l'expérience de pensée suivante met en lumière le problème: 

* Cluster A Amérique du Nord : flux massifs ($\mu$ très grand), ce qui génère mathématiquement une variancetrès grande à cause du terme $\mu^2$.  

* Cluster B Océanie : Des micro-flux ($\mu$ très bas), générant une variance minime.   


On concluerait (à tort) que le cluster A est beaucoup plus instable que le cluster B.   

Ici, $\phi$ élevé signifie cluster stable (Var = Esp) et vice-versa. C'est cela qu'on trace pour visualiser l'hétéroscédasticité, c'est même plus précis. De plus, on a maintenant une subdivision plus précise en régions M49 onusiennes. 



### LOO-CV en principe: 
Soustraire l'observation $y_i$ des données.  
Compiler HMC sur les N-1 obs restantes pour générer une distribution a posteriori $P(\theta | Y_{-i})$.  
Calculer la probabilité de prédire exactement l'obs exclue $y_i$ avec ces nouveaux paramètres : $P(y_i | \theta)$.

Avec 190*189 dyades (plus de 30 000) c'est physiquement irréalisable. 


Posterior global: $P(\theta | Y)$
Pour un $\theta^{(s)}$ donné qui comprend $\alpha^{(s)}, \gamma^{(s)}, \phi^{(s)}, \beta^{(s)}$ pour chaque itération $s$, Stan boucle sur $D_h$ (toutes les dyades) et calcule la vraisemblance locale $$\log p(y_i | \theta^{(s)}) = \log \text{ZTNB}(y_i | \theta^{(s)})$$ et génère un tenseur de dim itérations * chains * dyades 

BIC/AIC: vraisemblance en un point unique $\hat{\theta}$, ignore la complexité du paysage énergétique. 
Approche bayésienne: intégration des vraisemblances ZTNB locales sur tout l'espace, pondéré par la densité de l'espace des paramètres 
$$p(y_i | Y) = \int p(y_i | \theta) P(\theta | Y) d\theta \approx \frac{1}{S} \sum_{s=1}^S p(y_i | \theta^{(s)})$$



Approximation des paramètres effectifs:  

$$p_{\text{loo}} \approx \sum_{i=1}^N \text{Var}_{\text{post}}\left( \log p(y_i | \theta^{(s)}) \right)$$

- si un paramètre est contraint par un prior fort: sa vraisemblance ne changera pas beaucoup sur toutes les itérations. Variance proche de 0. 
- un paramètre peut avoir un poids>1


## Partie **PSIS**: 

Importance Sampling:  approxime le modèle privé de l'observation $i$ ($Y_{-i}$) en utilisant les paramètres du modèle complet ($Y$).

Soit $Y$ le vecteur des $N$ observations et $Y_{-i}$ le vecteur privé de la $i$-ème observation. 

hypothèse fondamentale des modèles hiérarchiques: : $$P(Y | \theta) = P(y_i | \theta) P(Y_{-i} | \theta)$$. 
Théorème de Bayes :$$P(\theta | Y) = \frac{P(Y | \theta) P(\theta)}{P(Y)} = \frac{P(y_i | \theta) P(Y_{-i} | \theta) P(\theta)}{P(Y)}$$
Distribution a posteriori (LOO): application de Bayes sur le sous-ensemble :$$P(\theta | Y_{-i}) = \frac{P(Y_{-i} | \theta) P(\theta)}{P(Y_{-i})}$$
Le poids d'importance d'un tirage $\theta^{(s)}$ est le quotient de ces deux densités : $$r_i^{(s)} = \frac{P(\theta^{(s)} | Y_{-i})}{P(\theta^{(s)} | Y)}$$.  
Substituons :$$r_i^{(s)} = \frac{\frac{P(Y_{-i} | \theta^{(s)}) P(\theta^{(s)})}{P(Y_{-i})}}{\frac{P(y_i | \theta^{(s)}) P(Y_{-i} | \theta^{(s)}) P(\theta^{(s)})}{P(Y)}}$$. 
Simplifions  :$$r_i^{(s)} = \frac{1}{P(y_i | \theta^{(s)})} \times \frac{P(Y)}{P(Y_{-i})}$$
Les termes $P(Y)$ et $P(Y_{-i})$ sont les vraisemblances marginales.  
Ce sont des constantes d'intégration invariantes par rapport à $\theta$. La dépendance du poids vis-à-vis des paramètres se réduit alors à : $$r_i^{(s)} \propto \frac{1}{P(y_i | \theta^{(s)})}$$

Le poids est inversement proportionnel à la vraisemblance locale des données. 



Distribution analysée: 

Le vecteur des $S=iter_sampling$ scalaires (empiriques) $\{r_i^{(1)}, r_i^{(2)}, \dots, r_i^{(S)}\}$ calculés pour une observation $i$ fixe 
On trie ces S poids, les 20% les plus lourds définissent la queue de la distribution, l'algorithme ajuste une GPD (generalized Pareto Distribution) dessus (theorème stochastique garantie la convergence de toute queue vers une GPD). la densité ressemble à ça, $$f(x | k, \sigma, \mu) = \frac{1}{\sigma} \left(1 + k \frac{x-\mu}{\sigma}\right)^{-(1/k + 1)}$$ et cela permet d'ajuster $k$ (paramètre de forme) et $\sigma$ (échelle).  

Si $k > 0.7$: L'esperance des poids n'existe pas,  queue trop épaisse, le lissage ne peut pas stabiliser la variance.    
Cela signifie que la distribution a posteriori change violemment lorsque l'on retire $y_i$.    
En d'autres termes, l'observation $i$ dicte massivement ses propres paramètres. Le modèle a overfitté cette observation précise.  
Exemple: Si Tuvalu n'a qu'un seul flux migratoire vers les USA, l'inférence du paramètre $\alpha_{TUV}$ repose uniquement sur cette observation $i$. Lorsque on exclut cette observation dans la LOO, Tuvalu n'a plus aucune donnée. Son $\alpha$ s'effondre brutalement vers le prior ignorant $\mu_{em}$. La différence entre le posterior complet et le posterior LOO est très grande ($k > 0.7$).   

C'est là qu'interviennent les priors structurels : si on retire l'unique flux de Tuvalu, son paramètre $\alpha_{TUV}$ ne chute plus dans le vide temporel d'une moyenne globale. Il est mécaniquement retenu par $Z_{TUV}\theta_{em}$ (sa population et son PIB). La macroéconomie empêche la distribution de se déformer brutalement. Le paramètre $k$ redescendra sous $0.7$ (à vérifier)   

Si $k \le 0.5$ : La variance des poids est finie. L'estimateur IS converge rapidement , (TCL).


On observe k>0.7 : il s'agit de substituer les hyper-moyennes ($\mu_{em}$, $\mu_{at}$) par une hyper-regression vectorielle $X\theta$


### Hyper-regression 
* $\mu_{em}$, $\mu_{at}$ non informatifs. Voici ce que fait Stan: 

vector[N_pays] alpha_em = mu_em + tau_em * alpha_em_raw;

vector[N_pays] gamma_at = mu_at + tau_at * gamma_at_raw;
* Exemple de Tuvalu: si une seule donnée retirée par la LOO-CV, son paramètre s'effondre dans le vide probabiliste vers le prior global $\mu_{em}$. 
Si peu de données mais des données quand même, l'attraction vers le prior reste forte, le paramètre $\alpha_{TUV}$ reste shrinké vers le prior non informatif. 


### MAE & MAPE comme uniques métriques de diagnostics et comparaison?


# à faire: 
# - mettre à jour le README technique 
- estimer BIC/AIC pour montrer que pénalise trop les paramètres
- PSIS-LOO sur le log normal (adapter le code, rendre fonctionnel) pour comparer NegBin em/at et Log Normale 
- préparer hyper-regression 
- biblio sur PSIS-LOO: très récent/moderne ? 



In [ ]:
import numpy as np
import arviz as az

# fichier compressé par arX_hurdle_NegBin.ipynb 

data_path = "./stan_outputs/log_lik_110pays_EmissionAttraction.npz"
with np.load(data_path) as data:
    log_lik_h = data['log_lik_h']
    log_lik_v = data['log_lik_v']

print(f"Shape Log-lik Volume: {log_lik_v.shape}")


idata_model = az.from_dict(
    log_likelihood={
        "hurdle": log_lik_h,
        "volume": log_lik_v
    }
)

# Calcul du critère PSIS-LOO
loo_result = az.loo(idata_model, var_name="volume")
print(loo_result)